# Computational Set: Data Analysis A — Linear Regression

### Calibration of a Pressure Gauge

**Learning objectives**

- Fit a linear data set $y = mx + b$ and extract the slope and intercept *together with their uncertainties*.
- Plot data and the regression line and judge the quality of a linear fit.
- Report parameters and uncertainties with correct units and significant figures.

---

#### Background

How do we know a gauge reads the correct value? We *calibrate* it against a known standard. Here a **capsule vacuum gauge** is calibrated against a **closed-end mercury manometer**, which we treat as the true pressure after a small correction for the thermal expansion of mercury:

$$
P(\text{Torr}) = P_{\text{obs}}\,\bigl(1 - 1.8\times10^{-4}\,T\bigr), \qquad T \text{ in } {}^{\circ}\mathrm{C}. \tag{1}
$$

The gauge response should be **linear** in pressure up to atmospheric pressure, so we model

$$
P = m\,P_{\text{gauge}} + b. \tag{2}
$$

In Excel you may have used the *Regression* tool to get $m$, $b$, and their standard deviations $S_m$, $S_b$. Here we do the same thing in Python with `numpy.polyfit`, which returns both the best-fit parameters **and** their covariance matrix — the square roots of its diagonal are exactly those standard deviations.


## New to Python? Start here

You don't need prior programming experience for this set. Almost everything below is
"fill in a short expression" using a handful of patterns. Here's all the syntax you'll need:

- **Comments.** Anything after a `#` is a note for humans; Python ignores it. The blanks you
  complete are marked with a comment like `# <-- replace with your expression`.

- **Variables.** `x = 5` stores the value `5` under the name `x`. Names are case-sensitive
  (`T` and `t` are different) and must be defined before you use them. Run cells **top to
  bottom** so each variable exists when later cells need it.

- **Arithmetic.** `+  -  *  /` work as expected, and `**` is *exponent* (so `x**2` is $x^2$).

- **Arrays (NumPy).** Our data lives in `np.array([...])`. The big convenience: arithmetic
  acts on the **whole array at once**. If `a` and `b` are arrays, `a - b` subtracts them
  element by element — no loop needed. Write the formula as if `a` and `b` were single numbers.

- **Calling a function.** A function takes inputs (*arguments*) in parentheses and often hands
  back a result you can store: `result = some_function(input1, input2)`. A function can return
  more than one value at once, which you catch with a comma: `a, b = gives_two_things()`.

- **Seeing output.** `print(...)` displays values so you can check your work.

> 💡 Want a proper, gentle introduction first? Work through the ESCIP
> **[“What is Python?” notebook](https://escip.io/notebooks/python/python-basics-fixed.html)**
> — it covers variables, functions, lists, and more, with chemistry examples. Highly
> recommended if any of the above is unfamiliar.

---


## The data

A run of the calibration produced, at each step:

- `P_dial` — the vacuum-gauge dial reading (inHg, read to 0.1 inHg),
- `h_left`, `h_right` — heights of the left and right mercury menisci (mm),

together with the barometric pressure `P_atm_inHg` and the manometer temperature `T_C`.

Run the cell below to load the data — **you don't need to edit it.**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Experimental constants ---
P_atm_inHg = 29.62     # barometric (atmospheric) pressure, inHg
T_C        = 22.5      # manometer temperature, degrees C

# --- Raw readings (one row per step) ---
P_dial  = np.array([29.2, 28.0, 26.5, 24.8, 23.2, 21.4, 19.9, 18.3, 16.7, 15.0])   # inHg
h_left  = np.array([749, 721, 680, 636, 596, 552, 513, 471, 429, 388], dtype=float) # mm
h_right = np.array([760, 759, 760, 760, 761, 761, 760, 760, 758, 761], dtype=float) # mm

print(f"{len(P_dial)} data points loaded.")

## Step 1 — Build the derived columns

In Excel you computed two new columns; here you build two NumPy arrays. NumPy applies an
arithmetic expression to every element of an array at once, so there is no need for a loop —
write the formula as if `P_dial`, `h_left`, and `h_right` were single numbers.

**The gauge pressure** (this will be the *x* data) is how far the true pressure sits above
the manifold's near-zero baseline, found by subtracting the dial reading from atmospheric
pressure, in inHg.

**The true pressure** (the *y* data) comes from the manometer. In a closed-end manometer the
pressure equals the *difference in height* of the two mercury columns. That raw height
difference is then scaled by the thermal-expansion factor of Eq. (1) to give a result in Torr.

👉 **Your task:** translate those two sentences into array expressions where indicated.


In [ ]:
# Thermal-expansion correction factor for mercury (the parenthesis in Eq. 1)
therm = 1 - 1.8e-4 * T_C

# x data, in inHg:  atmospheric pressure  -  dial reading
P_gauge =                  # <-- replace with your expression

# y data, in Torr:  (right column height  -  left column height)  scaled by  therm
P_torr =                   # <-- replace with your expression

print("P_gauge (inHg):", np.round(P_gauge, 2))
print("P_torr  (Torr):", np.round(P_torr, 1))

## Step 2 — Plot the data

Always look at the data before fitting, so you can judge whether a straight line is even
appropriate. We want an XY scatter plot of gauge pressure (horizontal axis) against true
pressure (vertical axis), drawn as **open markers with no connecting line**.

Matplotlib reminder: `ax.plot(x, y, 'o', mfc='none')` draws open circles; the third argument
is a *format string* (`'o'` = circles, `'-'` = a solid line).

👉 **Your task:** supply the `ax.plot(...)` call that draws the data points.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

# Draw the gauge pressure (x) against the true pressure (y) as open circles, no line.
ax.plot(                   )   # <-- pass x data, y data, and the format string

ax.set_xlabel('$P_{gauge}$ (inHg)')
ax.set_ylabel('$P$ (Torr)')
ax.tick_params(direction='in')
plt.show()

## Step 3 — Linear regression with uncertainties

`numpy.polyfit(x, y, deg, cov=True)` returns two things: the best-fit coefficients **and** the
covariance matrix of those coefficients.

- For a straight line use degree `deg = 1`.
- The coefficients come back **highest power first**, so the returned pair is
  `[slope, intercept]`.
- The standard deviations $S_m$ and $S_b$ are the **square roots of the diagonal** of the
  covariance matrix. `np.diag(cov)` pulls out that diagonal as an array.

👉 **Your task:** run the fit and pull out the two standard deviations.


In [ ]:
# Fit a degree-1 polynomial of the true pressure (y) on the gauge pressure (x),
# asking polyfit to also return the covariance matrix.
coeffs, cov =                       # <-- call np.polyfit(..., cov=True)

# coeffs is ordered [slope, intercept]
m, b = coeffs

# standard deviations = square root of the diagonal entries of the covariance matrix
S_m, S_b =                          # <-- take sqrt of the diagonal of cov

print(f"slope     m = {m:.4f}  (S_m = {S_m:.4f}) Torr/inHg")
print(f"intercept b = {b:.4f}  (S_b = {S_b:.4f}) Torr")

## Step 4 — Overlay the regression line

Using Eq. (2) with your fitted $m$ and $b$, compute the pressure the line *predicts* at every
gauge reading, then plot those predictions as a solid line on top of the data. A good fit
passes near every point.

👉 **Your task:** build the array of predicted pressures.


In [ ]:
# Predicted true pressure at each gauge reading:  slope * gauge pressure  +  intercept
P_calc =                   # <-- replace with your expression

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(P_gauge, P_torr, 'o', mfc='none', ms=6, label='data')
ax.plot(P_gauge, P_calc, '-', lw=1, label='regression line')
ax.set_xlabel('$P_{gauge}$ (inHg)')
ax.set_ylabel('$P$ (Torr)')
ax.tick_params(direction='in')
ax.legend()
plt.show()

## Step 5 — Report the parameters with correct significant figures

The lab's reporting convention:

- **Rule 1:** an uncertainty has **one** significant figure, *unless* its leading digit is 1 or
  2, in which case it keeps **two**.
- **Rule 2:** the last digit of the uncertainty sets the last digit reported in the value.

The `report` helper below encodes both rules. Run the cell to define it, then call it on each
parameter. It takes three arguments: the value, its uncertainty, and a unit string.

👉 **Your task:** call `report` for the slope and for the intercept.


In [ ]:
from math import log10, floor

# Format 'value +/- uncertainty unit' following the lab's two sig-fig rules.
def report(value, unc, unit=""):
    if unc == 0:
        return f"{value} {unit}"
    exp = floor(log10(abs(unc)))
    lead = int(abs(unc) / 10**exp)          # leading digit of the uncertainty
    sig = 2 if lead in (1, 2) else 1        # Rule 1
    dp = -(exp - (sig - 1))                 # decimal places to keep
    if dp >= 0:
        return f"{value:.{dp}f} ± {unc:.{dp}f} {unit}"
    f = 10**(-dp)
    return f"{round(value/f)*f:g} ± {round(unc/f)*f:g} {unit}"

# Report each parameter as  value ± uncertainty  with its unit.
print("m =", report(            ))   # <-- pass slope, its uncertainty, "Torr/inHg"
print("b =", report(            ))   # <-- pass intercept, its uncertainty, "Torr"


## Discussion

Answer briefly in the cell below (as Markdown):

1. Do the data look linear? How well does the regression line follow them?
2. One inch of mercury equals 25.40 Torr. Compare your fitted slope (with its uncertainty) to this expected value — are they consistent?
3. What does the intercept tell you physically, and is it consistent with zero within its uncertainty?


*Your answers here.*